# 09 · TotalSpineSeg en Kaggle para las series lumbares de Mendeley (cuaderno auxiliar)

Algunos pacientes de Mendeley tienen en su carpeta series sagitales de la columna cervical y de la lumbar con la misma descripción, y en 21 estudios la serie elegida en el cuaderno 08 es cervical. El cuaderno 09 arma la lista `lista_mendeley_lumbares.csv` con las demás series sagitales de esa secuencia en esos estudios, y este cuaderno las segmenta con TotalSpineSeg, con las mismas versiones y pesos de los otros cuadernos auxiliares. Después, el cuaderno 09 elige en cada estudio la serie que trae los discos lumbares.

**Antes de correrlo, en Kaggle:**

1. Sube como dataset privado el zip `kaggle_entrada_lumbares.zip` que deja la celda 6 del cuaderno 09 en `content/datos/segmentacion_tss/mendeley/` (*Datasets → New Dataset*, con el nombre **prl-09-mendeley-lumbar**). Trae `lista_mendeley_lumbares.csv` y los NIfTI de las series candidatas.
2. *Settings → Accelerator*: **GPU T4 x2**. *Settings → Internet*: **On**. *Add Input*: el dataset **prl-09-mendeley-lumbar**.
3. *Save Version → Save & Run All (Commit)*. Son unas 44 series, así que debería tomar del orden de 15 a 20 minutos, la mayor parte en instalar TotalSpineSeg.

Al terminar, descarga la salida y deja sus archivos en `content/datos/segmentacion_tss/mendeley/` de Drive, que es donde los lee el cuaderno 09.

In [ ]:
# 1. Entorno aparte para TotalSpineSeg (igual que en el cuaderno 05)
import subprocess, sys

def correr(cmd, **kw):
    "Corre un comando y, si falla, muestra su salida de error antes de detenerse."
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Falló: {cmd if isinstance(cmd, str) else ' '.join(cmd)}")
    return r

correr("pip -q install virtualenv SimpleITK", shell=True)
correr("rm -rf /kaggle/tmp/venv_tss && virtualenv -q --system-site-packages /kaggle/tmp/venv_tss", shell=True)
correr(["/kaggle/tmp/venv_tss/bin/pip", "-q", "install", "totalspineseg==20260807", "nnunetv2==2.6.2", "kornia==0.8.2"])
r = correr(["/kaggle/tmp/venv_tss/bin/python", "-c",
            "import torch; from importlib.metadata import version as v; "
            "print(torch.__version__, torch.cuda.device_count(), v('totalspineseg'))"])
version_torch, gpus_tss, version_tss = r.stdout.split()
print(f"torch {version_torch} | GPUs: {gpus_tss} | totalspineseg {version_tss}")
assert int(gpus_tss) >= 1, "TotalSpineSeg no ve ninguna GPU: revisa Settings → Accelerator antes de seguir"
assert version_tss == "20260807", "Versión de TotalSpineSeg distinta de la fijada"

In [ ]:
# 2. Parámetros y rutas
LISTA = "lumbares"
INICIO = 1
HORAS_MAXIMAS = 11.0
TAM_LOTE = 20

import os, time, json, shutil, hashlib, datetime, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk

ENTRADA = Path("/kaggle/input")
TRABAJO = Path("/kaggle/working")
TMP = Path("/kaggle/tmp")
VENV_TSS = TMP / "venv_tss"
DIR_TSS_DATOS = TMP / "tss_datos"
DIR_MASCARAS = TMP / "mascaras"
for d in (DIR_TSS_DATOS, DIR_MASCARAS):
    d.mkdir(parents=True, exist_ok=True)
CARPETAS_DE_IMAGENES = {"train_images", "test_images"}

def buscar(patron):
    "Archivos de las entradas que cumplen el patrón, sin entrar en carpetas de imágenes de competiciones."
    hallados = []
    for raiz, dirs, archivos in os.walk(ENTRADA):
        dirs[:] = [d for d in dirs if d not in CARPETAS_DE_IMAGENES]
        hallados += [Path(raiz) / a for a in archivos if Path(a).match(patron)]
    return sorted(hallados)

RUTA_LISTA = buscar("lista_mendeley_lumbares.csv")
assert RUTA_LISTA, "No se encontró lista_mendeley_lumbares.csv: agrega el dataset prl-09-mendeley-lumbar como entrada"
lista = pd.read_csv(RUTA_LISTA[0], dtype={"estudio": str, "serie_clave": str})
# Kaggle descomprime los .gz al procesar un dataset, así que los NIfTI pueden llegar como .nii en vez de .nii.gz
NIFTI = {p.name.split(".nii")[0]: p for p in buscar("men_*.nii*")}
faltan = sorted(set(lista["id_serie"]) - set(NIFTI))
assert not faltan, f"Faltan {len(faltan)} NIfTI de la lista en las entradas: {faltan[:5]}"
print(f"Series lumbares candidatas: {len(lista)} series de {lista['estudio'].nunique()} estudios; NIfTI encontrados: {len(NIFTI)}")

# Las mismas comprobaciones de la conversión del cuaderno 08
TOL_GEOMETRIA = 1e-3
def misma_geometria(a, b, tol=TOL_GEOMETRIA):
    "True si dos imágenes ocupan exactamente la misma rejilla en el espacio."
    return (a.GetSize() == b.GetSize() and np.allclose(a.GetSpacing(), b.GetSpacing(), atol=tol)
            and np.allclose(a.GetOrigin(), b.GetOrigin(), atol=tol)
            and np.allclose(a.GetDirection(), b.GetDirection(), atol=tol))

def huella_volumen(img):
    "md5 de los vóxeles y de la geometría redondeada (igual que en el cuaderno 08)."
    h = hashlib.md5(np.ascontiguousarray(sitk.GetArrayViewFromImage(img)).tobytes())
    geometria = (img.GetSize(), tuple(round(v, 4) for v in img.GetSpacing()),
                 tuple(round(v, 3) for v in img.GetOrigin()), tuple(round(v, 5) for v in img.GetDirection()))
    h.update(repr(geometria).encode())
    return h.hexdigest()

def geometria_deformada(img):
    return bool(np.allclose(img.GetSpacing(), 1.0) and np.allclose(img.GetOrigin(), 0.0))

prueba = sitk.ReadImage(str(NIFTI[lista["id_serie"].iloc[0]]))
print("Primera serie: espaciado", np.round(prueba.GetSpacing(), 3), "| origen", np.round(prueba.GetOrigin(), 1))
assert not geometria_deformada(prueba), "La primera serie tiene espaciado 1 × 1 × 1 y origen en cero: no seguir"

In [ ]:
# 3. Pesos de TotalSpineSeg de la release fijada, verificados antes de instalarlos
import requests
RELEASE_PESOS_TSS = "r20260730"
SHA256_PESOS_TSS = {
    "Dataset101_TotalSpineSeg_step1_r20260730.zip": "781f81e8c94feccd3cef17d819ba5c293b506e9b1f2b106fc5e31b9dd85887b0",
    "Dataset102_TotalSpineSeg_step2_r20260730.zip": "494c640ab8cb25475bd0fc4338c0931d37079af9dd19403879335c3908c06b46",
}
def sha256_archivo(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()

os.environ["TOTALSPINESEG_DATA"] = str(DIR_TSS_DATOS)
dir_exports = DIR_TSS_DATOS / "nnUNet" / "exports"
dir_exports.mkdir(parents=True, exist_ok=True)
for nombre, sha_esperado in SHA256_PESOS_TSS.items():
    destino = dir_exports / nombre
    if not destino.exists():
        with requests.get(f"https://github.com/neuropoly/totalspineseg/releases/download/{RELEASE_PESOS_TSS}/{nombre}",
                          stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(destino, "wb") as f:
                for trozo in r.iter_content(1 << 20):
                    f.write(trozo)
    assert sha256_archivo(destino) == sha_esperado, f"{nombre} NO coincide con la huella verificada; no continuar."
subprocess.run([str(VENV_TSS / "bin" / "totalspineseg_init"), "--quiet"], check=True, env=os.environ.copy())
print("Pesos verificados e instalados")

In [ ]:
# 4. Segmentación por lotes, en las GPU disponibles a la vez
import torch
N_GPU = max(torch.cuda.device_count(), 1)
CSV_REG = TRABAJO / "registro_mendeley_lumbares.csv"
ZIP_MASCARAS = TRABAJO / "mascaras_mendeley_lumbares.zip"
COLUMNAS_REG = ["lista", "orden", "conjunto", "id_serie", "estado", "geometria_ok", "huella_ok", "segundos_lote",
                "series_en_lote", "gpu", "fecha", "error"]
pendientes = lista.sort_values("orden")

def lanzar(lote, gpu):
    base = TMP / f"lote_gpu{gpu}"
    shutil.rmtree(base, ignore_errors=True)
    (base / "entrada").mkdir(parents=True)
    huellas = {}
    for r in lote.itertuples():
        try:
            img = sitk.ReadImage(str(NIFTI[r.id_serie]))            # puede venir como .nii: se reescribe como .nii.gz
            sitk.WriteImage(img, str(base / "entrada" / f"{r.id_serie}.nii.gz"), useCompression=True)
            huellas[r.id_serie] = huella_volumen(img) == r.huella_volumen
        except Exception as e:
            huellas[r.id_serie] = f"error de entrada: {type(e).__name__}: {e}"[:200]
    cmd = [str(VENV_TSS / "bin" / "totalspineseg"), str(base / "entrada"), str(base / "salida"),
           "--device", "cuda", "--keep-only", "step2_output", "--quiet"]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "MPLBACKEND": "Agg"}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=env)
    return {"lote": lote, "gpu": gpu, "base": base, "huellas": huellas, "proc": proc, "t0": time.time()}

def mensaje_de_error(texto):
    lineas = [l.strip() for l in texto.replace("\r", "\n").splitlines()]
    utiles = [l for l in lineas if l and "it/s]" not in l and "s/it]" not in l and "%|" not in l]
    return " | ".join(utiles)[-300:]

def recoger(trabajo):
    salida, error = trabajo["proc"].communicate()
    dt = time.time() - trabajo["t0"]
    filas, ok = [], 0
    for r in trabajo["lote"].itertuples():
        fila = dict.fromkeys(COLUMNAS_REG, "")
        fila.update({"lista": LISTA, "orden": r.orden, "conjunto": r.conjunto, "id_serie": r.id_serie,
                     "segundos_lote": round(dt, 1), "series_en_lote": len(trabajo["lote"]), "gpu": trabajo["gpu"],
                     "fecha": datetime.datetime.now().isoformat(timespec="seconds")})
        huella = trabajo["huellas"][r.id_serie]
        out = trabajo["base"] / "salida" / "step2_output" / f"{r.id_serie}.nii.gz"
        if isinstance(huella, str):
            fila.update({"estado": "error", "error": huella})
        elif out.exists():
            img = sitk.ReadImage(str(trabajo["base"] / "entrada" / f"{r.id_serie}.nii.gz"))
            seg = sitk.ReadImage(str(out))
            geometria_ok = misma_geometria(img, seg)
            if geometria_ok:
                seg.CopyInformation(img)
            local = DIR_MASCARAS / f"{r.id_serie}.nii.gz"
            sitk.WriteImage(sitk.Cast(seg, sitk.sitkUInt8), str(local), useCompression=True)
            with zipfile.ZipFile(ZIP_MASCARAS, "a") as z:
                z.write(local, local.name)
            local.unlink()
            fila.update({"estado": "ok", "geometria_ok": geometria_ok, "huella_ok": huella})
            ok += 1
        else:
            fila.update({"estado": "error", "error": "TotalSpineSeg sin salida: " + mensaje_de_error(error)})
        filas.append(fila)
    pd.DataFrame(filas, columns=COLUMNAS_REG).to_csv(CSV_REG, mode="a", header=not CSV_REG.exists(), index=False)
    return ok, dt

lotes = [pendientes.iloc[i:i + TAM_LOTE] for i in range(0, len(pendientes), TAM_LOTE)]
inicio, siguiente, en_curso, total_ok = time.time(), 0, [], 0
while siguiente < len(lotes) or en_curso:
    libres = [g for g in range(N_GPU) if g not in {t["gpu"] for t in en_curso}]
    while libres and siguiente < len(lotes):
        en_curso.append(lanzar(lotes[siguiente], libres.pop(0)))
        siguiente += 1
    terminado = None
    while terminado is None:
        for t in en_curso:
            if t["proc"].poll() is not None:
                terminado = t
                break
        else:
            time.sleep(5)
    en_curso = [t for t in en_curso if t is not terminado]
    ok, dt = recoger(terminado)
    total_ok += ok
    print(f"Lote en GPU {terminado['gpu']}: {ok}/{len(terminado['lote'])} ok en {dt:.0f} s | {total_ok} máscaras")
    if ok == 0 and total_ok == 0:
        raise RuntimeError("El primer lote no produjo ninguna máscara: revisar el registro")

In [ ]:
# 5. Comprobación final y registro del entorno
from importlib.metadata import version as v
reg = pd.read_csv(CSV_REG, dtype=str)
with zipfile.ZipFile(ZIP_MASCARAS) as z:
    en_zip = {n.split(".nii")[0] for n in z.namelist()}
ok_registro = set(reg.loc[reg["estado"] == "ok", "id_serie"])
print(f"Máscaras en {ZIP_MASCARAS.name}: {len(en_zip)} | correctas en el registro: {len(ok_registro)} de {len(lista)} | "
      f"sin máscara en el zip: {len(ok_registro - en_zip)} | huellas distintas: {(reg['huella_ok'] == 'False').sum()} | "
      f"geometría distinta: {(reg['geometria_ok'] == 'False').sum()}")
print("Errores:", reg.loc[reg["estado"] == "error", ["id_serie", "error"]].to_string(index=False) if (reg["estado"] == "error").any() else "ninguno")
entorno = {"fecha": datetime.datetime.now().isoformat(timespec="seconds"), "lista": LISTA, "inicio": INICIO,
           "corrida": "series lumbares de Mendeley que reemplazan a las cervicales", "python": sys.version.split()[0],
           "gpus": N_GPU, "gpu_nombre": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "totalspineseg": "20260807", "pesos_release": RELEASE_PESOS_TSS, "pesos_sha256": SHA256_PESOS_TSS,
           "tam_lote": TAM_LOTE, "simpleitk": v("SimpleITK")}
(TRABAJO / "entorno_mendeley_lumbares.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False))
print(json.dumps(entorno, indent=2, ensure_ascii=False))